# PyroFinder EDA Notebook

**Course:** Technion 016833 — Location-Based Services: Data Science  
**Milestone:** M2 checkpoint (26/05/2026)  
**Dataset:** D-Fire (primary)  

**Prerequisites:** Generate the metadata CSV first:
```bash
python scripts/build_dfire_metadata.py \
  --raw-root "C:\Users\boris.azarov\OneDrive - Technion\Desktop\PyroFinder\RAW_DATA\D-Fire" \
  --output data/dfire_metadata.csv
```

This notebook runs without raw images as long as `data/dfire_metadata.csv` exists.

## 0. Imports and Configuration

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Allow imports from project root
sys.path.insert(0, str(Path.cwd().parent))

from src.data import load_dfire_metadata, clean_dfire_metadata, get_dfire_summary
from src.eda import (
    compute_summary_metrics,
    compute_category_counts,
    compute_split_counts,
    compute_bbox_stats,
    get_primary_eda_insight,
)

METADATA_PATH = Path.cwd().parent / "data" / "dfire_metadata.csv"
print("Metadata CSV path:", METADATA_PATH)
print("Exists:", METADATA_PATH.exists())

## 1. Load and Inspect Metadata

In [ ]:
raw_df = load_dfire_metadata(METADATA_PATH)
df = clean_dfire_metadata(raw_df)
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 2. Summary Metrics

In [ ]:
summary = get_dfire_summary(df)
for k, v in summary.items():
    print(f"{k}: {v}")

## 3. Image Category Distribution

In [ ]:
print("image_category counts:")
print(df["image_category"].value_counts())

In [ ]:
cat_df = compute_category_counts(df)

fig = px.bar(
    cat_df,
    x="category",
    y="count",
    color="category",
    color_discrete_map={
        "fire_only": "#e05c00",
        "smoke_only": "#7b9ccc",
        "fire_and_smoke": "#c44b00",
        "background": "#888888",
    },
    title="D-Fire: Image category distribution",
    labels={"category": "Category", "count": "Images"},
)
fig.update_layout(showlegend=False)
fig.show()

## 4. Split Distribution

In [ ]:
print("Split counts:")
print(df["split"].value_counts())

In [ ]:
split_df = compute_split_counts(df)

fig2 = px.bar(
    split_df,
    x="split",
    y="count",
    color="split",
    title="D-Fire: Images per split",
    labels={"split": "Split", "count": "Images"},
)
fig2.update_layout(showlegend=False)
fig2.show()

## 5. Bounding Box Statistics

In [ ]:
bbox_stats = compute_bbox_stats(df)
for k, v in bbox_stats.items():
    print(f"{k}: {v:.4f}")

In [ ]:
labeled = df[df["total_boxes"] > 0]
print(f"Images with at least one box: {len(labeled)} of {len(df)}")

fig3 = px.histogram(
    labeled,
    x="total_boxes",
    nbins=30,
    color_discrete_sequence=["#e05c00"],
    title="Distribution of bounding box count per image (labeled images only)",
    labels={"total_boxes": "Total boxes", "count": "Images"},
)
fig3.show()

## 6. EDA Insight

In [ ]:
insight = get_primary_eda_insight(df)
print(insight)

## 7. Future Work (M3 and beyond)

The following items are **intentionally not done** in M2 and are planned for M3:

- **YOLO11n baseline run:** `model.val(data=..., split='val')` to get mAP@0.5 without fine-tuning.
- **YOLO11s fine-tuning:** `model.train(data=..., epochs=50, imgsz=640)` on D-Fire train split.
- **YOLO11s vs YOLO11n comparison:** side-by-side mAP, precision, recall, inference speed.
- **Deployment / live RTSP ingestion:** out of scope this semester.
- **Additional datasets:** HPWREN, FURG, Smart Fire — planned for M3+.

```python
# M3 placeholder — do not run in M2
# from ultralytics import YOLO
# model_n = YOLO("yolo11n.pt")
# results = model_n.val(data="data/dfire/dataset.yaml", split="val")
# print("YOLO11n mAP@0.5:", results.box.map50)
```

---
*PyroFinder · Technion Course 016833 · Location-Based Services: Data Science · 2026*